<a href="https://colab.research.google.com/github/DiaaEssam/Artificial-Neural-Network-from-scratch-applied-on-MNIST/blob/main/ANN_from_scratch_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div class="alert alert-success" style = "border-radius: 20px;text-align: center;" role="alert">
    Don't forget to upvote if you liked the notebook
</div>

# Importing Libraries

In [ ]:
import numpy as np
import pandas as pd
from keras.datasets import mnist
import tensorflow as tf
from keras.datasets import mnist
import tensorflow.keras.layers as tfl
np.seterr(over='ignore')

# Loading MNIST Dataset

In [ ]:
#(X_train, y_train), (X_test, y_test) = mnist.load_data()
X_train=pd.read_csv("/kaggle/input/digit-recognizer/train.csv")
X_test=pd.read_csv("/kaggle/input/digit-recognizer/test.csv")
image_id=pd.Series(X_test.iloc[:,0])
df = image_id.reset_index()
image_id = df['index']
X_train=np.array(X_train)
X_test=np.array(X_test)

# Loading y_train

In [ ]:
y_train=X_train[:,0]
print(y_train)

# Deleting y_train from Train data 

In [ ]:
X_train=np.delete(X_train, 0, 1)
print(X_train[:,0])

# Applying One Hot Encodeing for labels

In [ ]:
"""num_classes=len(np.unique(y_train))
y_train=np.array([np.insert(np.zeros(num_classes-1),i,1) for i in y_train]).astype(int) another way implemented by me"""

from sklearn.preprocessing import OneHotEncoder

enc = OneHotEncoder()
y_train=enc.fit_transform(y_train.reshape(y_train.shape[0],1)).toarray().astype(int)

print(y_train.shape)
print(y_train[0])

In [ ]:
X_train = X_train.astype('float32')
X_test = X_test.astype('float32')

In [ ]:
X_train=X_train.reshape(X_train.shape[0],28,28)
X_test=X_test.reshape(X_test.shape[0],28,28)

print('X_train: ' + str(X_train.shape))
print('Y_train: ' + str(y_train.shape))
print('X_test:  '  + str(X_test.shape))

# Normalizing The Data

In [ ]:
def normalize(X):
    return X/255.0 # we need epsilon for feature (pixel) that has zero variance

In [ ]:
X_train=normalize(X_train)
X_test=normalize(X_test)

X_train=X_train.reshape(X_train.shape[0],X_train.shape[1],X_train.shape[2],1)
X_test=X_test.reshape(X_test.shape[0],X_test.shape[1],X_test.shape[2],1)

# Forward Propagation (Le Net 5 architecture)

In [ ]:

def arch(input_shape):
    
    input_img = tf.keras.Input(shape=input_shape)
    Z1=tfl.Conv2D(filters= 6 , kernel_size= 5,strides=(1, 1))(input_img)
    A1=tfl.ReLU()(Z1)
    P1=tfl.AveragePooling2D(pool_size=(2, 2), strides=(2, 2))(A1)
    BT1=tfl.BatchNormalization()(P1)

    Z2=tfl.Conv2D(filters= 16 , kernel_size= 5 ,strides=(1, 1))(BT1)
    A2=tfl.ReLU()(Z2)
    P2=tfl.AveragePooling2D(pool_size=(5, 5), strides=(2,2))(A2)
    BT2=tfl.BatchNormalization()(P2)

    F1=tfl.Flatten()(BT2)

    FC1=tfl.Dense(units=120, activation='relu')(F1)
    D1=tfl.Dropout(0.4)(FC1)
    BT3=tfl.BatchNormalization()(D1)

    FC2=tfl.Dense(units=84, activation='relu')(BT3)
    D2=tfl.Dropout(0.4)(FC2)
    BT4=tfl.BatchNormalization()(D2)

    outputs=tfl.Dense(units= 10 , activation='softmax')(BT4)
    model = tf.keras.Model(inputs=input_img, outputs=outputs)
    return model

# Defining the model

In [ ]:
conv_model = arch((28, 28, 1))
conv_model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
conv_model.summary()

# Training

In [ ]:
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train)).batch(64)
history = conv_model.fit(train_dataset, epochs=200)

# Making predictions

In [ ]:
y_pred=conv_model.predict(X_test)
y_pred=np.argmax(y_pred,axis=1)

# Submitting

In [ ]:
submission = pd.DataFrame({
         "ImageId": range(1, len(X_test) + 1),
         "Label": y_pred
     })
submission.to_csv('submission.csv', index=False)